# A GPU muda o resultado? — teste L4 × T4

Responde duas perguntas de uma vez:

1. **Os resultados divergem entre GPUs?** Se sim, a partir de qual casa decimal,
   em qual camada, e com que velocidade a diferença cresce durante o treino.
2. **Quanto custa pinar tudo numa GPU só?** A razão de velocidade L4/T4 nesta
   carga, que é o que decide se vale abrir mão das vagas de T4 no painel.

## Como usar

1. **Ambiente de execução → Alterar tipo de ambiente → L4.** Execute tudo.
2. Copie o bloco `COLE ESTE BLOCO` impresso no fim.
3. **Altere para T4.** Execute tudo de novo. Copie o segundo bloco.
4. Cole os dois na última célula e execute só ela.

Leva cerca de 2-4 min por GPU.

## O que este teste cobre e o que não cobre

As flags de determinismo, a arquitetura (4 camadas convolucionais, entrada
`2×1024`, lote 64) e o formato dos tensores são **os mesmos** de
`campanha/busca_hp.py`, então os mesmos kernels do cuDNN são exercitados.

Os **dados são sintéticos**, não o RadioML. Isso é de propósito: torna o
notebook autocontido e rápido, e a ordem de redução do cuDNN — que é a fonte da
divergência — não depende do conteúdo dos dados, só do formato. O que este teste
**não** mede é se um veredito de colapso específico vira de vivo para morto;
para isso seria preciso o dataset real numa run em cima do muro.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURAÇÃO — não precisa mexer para o teste padrão
# ══════════════════════════════════════════════════════════════════════
SEMENTE     = 42        # idêntica nas duas GPUs, obviamente
N_TREINO    = 8000
N_VAL       = 2000
LOTE        = 64
EPOCAS      = 12        # o bastante para a divergência ficar macroscópica
LR          = 4.5e-5    # o mesmo default da campanha
PASSOS_LOG  = 30        # perdas individuais registradas no início do treino

# Dificuldade da tarefa sintética. Calibrado na CPU: leva a acurácia de
# ~39% para ~56% em 10 épocas, sem saturar. Saturar em 100% cegaria a
# Fase B -- se as duas GPUs acertam tudo, a acurácia não mostra divergência.
RUIDO       = 1.0       # desvio do ruído aditivo
PASSO_F     = 0.5       # separação de frequência entre classes vizinhas
SIGMA_R     = 0.8       # jitter da frequência, em unidades de PASSO_F
EPOCAS_TEMPO = 5        # épocas cronometradas para a razão de velocidade

# Espelha campanha/busca_hp.py:476-483. Mudar isto invalida a comparação
# com a campanha.
DETERMINISTA = True
TF32         = False    # a L4 ligaria por padrão; a T4 nem tem. Ver o notebook.

In [ ]:
import json, math, time, platform
import numpy as np
import torch, torch.nn as nn

torch.backends.cudnn.deterministic = DETERMINISTA
torch.backends.cudnn.benchmark     = not DETERMINISTA
torch.backends.cudnn.allow_tf32       = TF32
torch.backends.cuda.matmul.allow_tf32 = TF32

assert torch.cuda.is_available(), "sem GPU: Ambiente de execução → Alterar tipo"
DEV  = torch.device("cuda")
INFO = {
    "gpu"        : torch.cuda.get_device_name(0),
    "capability" : "%d.%d" % torch.cuda.get_device_capability(0),
    "sms"        : torch.cuda.get_device_properties(0).multi_processor_count,
    "torch"      : torch.__version__,
    "cuda"       : torch.version.cuda,
    "cudnn"      : torch.backends.cudnn.version(),
    "driver"     : platform.uname().release,
    "determinist": DETERMINISTA,
    "tf32"       : bool(torch.backends.cudnn.allow_tf32),
}
for k, v in INFO.items():
    print("%-12s %s" % (k, v))

In [ ]:
# Dados sintéticos com estrutura tipo-modulação. As classes se
# SOBREPÕEM de propósito: a frequência de cada classe leva jitter gaussiano de
# desvio SIGMA_R*PASSO_F, e a fase é aleatória por amostra (não é pista de
# classe). Isso cria um teto de Bayes bem abaixo de 100% e mantém a acurácia
# numa faixa onde ela ainda consegue discriminar.
#
# Gerados na CPU com semente fixa do numpy: saem bit a bit iguais nas duas GPUs.
N_CLASSES = 5
COMPRIMENTO = 1024

def gera(n, semente):
    rng = np.random.default_rng(semente)
    y = rng.integers(0, N_CLASSES, size=n)
    t = np.arange(COMPRIMENTO, dtype=np.float32) / COMPRIMENTO
    freq = (1.0 + y * PASSO_F
            + rng.normal(0, SIGMA_R * PASSO_F, size=n)).astype(np.float32)
    fase = rng.uniform(0, 2 * np.pi, size=n).astype(np.float32)
    amp  = rng.standard_normal((n, 1)).astype(np.float32) * 0.1 + 1.0
    arg  = 2 * np.pi * freq[:, None] * t[None, :] + fase[:, None]
    i = (amp * np.cos(arg)).astype(np.float32)
    q = (amp * np.sin(arg)).astype(np.float32)
    x = np.stack([i, q], axis=1)
    x += rng.standard_normal(x.shape).astype(np.float32) * RUIDO
    return torch.from_numpy(x), torch.from_numpy(y.astype(np.int64))

XTR, YTR = gera(N_TREINO, SEMENTE)
XVA, YVA = gera(N_VAL, SEMENTE + 1000)

# Ordem dos lotes fixada por permutação do numpy: sem DataLoader, sem
# gerador de RNG, nenhuma fonte de variação na ordem dos dados.
ORDEM = [np.random.default_rng(SEMENTE + 1 + e).permutation(N_TREINO)
         for e in range(max(EPOCAS, EPOCAS_TEMPO) + 1)]

print("treino", tuple(XTR.shape), "| val", tuple(XVA.shape))
print("chance = %.1f%% | esperado ao fim: ~55%%" % (100.0 / N_CLASSES))
print("checksum dos dados (float64):", float(XTR.double().sum()).hex())

In [ ]:
# A mesma arquitetura 4L da referência do projeto (escolhe_lr.py:81-84).
ARQ = [{"out_channels": 32,  "kernel_size": 11},
       {"out_channels": 64,  "kernel_size": 7},
       {"out_channels": 128, "kernel_size": 5},
       {"out_channels": 256, "kernel_size": 3}]

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        camadas, ch = [], 2
        for b in ARQ:
            ks = b["kernel_size"]
            camadas += [nn.Conv1d(ch, b["out_channels"], ks, padding=ks // 2),
                        nn.BatchNorm1d(b["out_channels"]), nn.ReLU(),
                        nn.MaxPool1d(2)]
            ch = b["out_channels"]
        self.features = nn.Sequential(*camadas)
        self.flatten  = nn.Flatten()
        n = self.flatten(self.features(torch.zeros(1, 2, COMPRIMENTO))).size(1)
        self.classifier = nn.Sequential(nn.Linear(n, 512), nn.ReLU(),
                                        nn.Dropout(0.5), nn.Linear(512, N_CLASSES))
    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))

def novo_modelo():
    """Inicializa na CPU com semente fixa, depois move. Garante pesos
    idênticos nas duas GPUs antes de qualquer aritmética de GPU."""
    torch.manual_seed(SEMENTE)
    np.random.seed(SEMENTE)
    m = CNN()
    return m.to(DEV)

_m = novo_modelo()
print("parâmetros:", "{:,}".format(sum(p.numel() for p in _m.parameters())))
print("checksum dos pesos iniciais (CPU, float64):",
      float(sum(p.detach().cpu().double().sum() for p in _m.parameters())).hex())
del _m

In [ ]:
# ── FASE A ── aritmética pura, sem treino ────────────────────────────
# Um único forward com pesos idênticos. Sem otimizador, sem caos: qualquer
# diferença aqui é o kernel do cuDNN e mais nada. Os ganchos por camada
# localizam ONDE a divergência nasce.

def impressao_digital():
    m = novo_modelo().eval()
    saidas = {}
    ganchos = []
    for nome, mod in m.features.named_children():
        def cria(n, mo):
            def gancho(_mod, _ent, sai):
                saidas["features.%s.%s" % (n, type(mo).__name__)] = \
                    float(sai.double().sum())
            return gancho
        ganchos.append(mod.register_forward_hook(cria(nome, mod)))
    x = XTR[:LOTE].to(DEV)
    with torch.no_grad():
        logits = m(x)
    for g in ganchos:
        g.remove()
    saidas["logits"] = float(logits.double().sum())
    return {k: v.hex() for k, v in saidas.items()}, logits

A1, lg1 = impressao_digital()
A2, _   = impressao_digital()

igual = (A1 == A2)
print("repetição na MESMA GPU:", "idêntica" if igual else "DIFERENTE (!)")
if not igual:
    print("  as flags de determinismo não bastam nesta máquina;")
    print("  a comparação entre GPUs fica sem linha de base.")
print()
for k, v in A1.items():
    print("  %-34s %s" % (k, v))

In [ ]:
# ── FASE B ── quanto a diferença cresce durante o treino ─────────────
def treina(n_epocas, cronometra=False):
    m = novo_modelo()
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    passos, epocas, tempos = [], [], []
    xva, yva = XVA.to(DEV), YVA.to(DEV)
    for e in range(n_epocas):
        m.train()
        torch.cuda.synchronize(); t0 = time.perf_counter()
        soma, nb = 0.0, 0
        for i in range(0, N_TREINO - LOTE + 1, LOTE):
            idx = ORDEM[e][i:i + LOTE]
            xb = XTR[idx].to(DEV, non_blocking=False)
            yb = YTR[idx].to(DEV, non_blocking=False)
            opt.zero_grad(set_to_none=True)
            perda = crit(m(xb), yb)
            perda.backward(); opt.step()
            if e == 0 and len(passos) < PASSOS_LOG:
                passos.append(float(perda.detach().double()).hex())
            soma += float(perda.detach()); nb += 1
        torch.cuda.synchronize()
        tempos.append(time.perf_counter() - t0)
        m.eval()
        with torch.no_grad():
            acc = float((m(xva).argmax(1) == yva).float().mean()) * 100
        pesos = float(sum(p.detach().cpu().double().sum() for p in m.parameters()))
        epocas.append({"ep": e + 1, "perda": (soma / nb).hex(),
                       "acc": acc, "pesos": pesos.hex(),
                       "s": round(tempos[-1], 4)})
        if not cronometra:
            print("  ep %2d | perda %.10f | acc %6.2f%% | %.2fs"
                  % (e + 1, soma / nb, acc, tempos[-1]))
    return passos, epocas, tempos

print("treino de %d épocas:" % EPOCAS)
PASSOS, EPOCAS_HIST, _ = treina(EPOCAS)

In [ ]:
# ── FASE C ── velocidade, para dimensionar o custo de pinar ──────────
# A primeira época carrega o custo de alocação e autotune; descartada.
_, _, TEMPOS = treina(EPOCAS_TEMPO, cronometra=True)
UTEIS = TEMPOS[1:]
SEG_EPOCA = sum(UTEIS) / len(UTEIS)
print("épocas cronometradas: %s" % ["%.3f" % t for t in UTEIS])
print("mediana por época: %.3f s" % sorted(UTEIS)[len(UTEIS) // 2])
print("média por época:   %.3f s" % SEG_EPOCA)

In [ ]:
RESULTADO = {"info": INFO, "mesma_gpu_identica": igual,
             "fase_a": A1, "passos": PASSOS, "epocas": EPOCAS_HIST,
             "seg_epoca": SEG_EPOCA}

nome = "/content/teste_gpu_%s.json" % INFO["gpu"].replace(" ", "_")
with open(nome, "w") as f:
    json.dump(RESULTADO, f, indent=1)
print("salvo em", nome)
print()
print("=" * 70)
print("COLE ESTE BLOCO (é uma linha só)")
print("=" * 70)
print(json.dumps(RESULTADO, separators=(",", ":")))
print("=" * 70)

## Comparação

Cole aqui os dois blocos e execute **só esta célula**. Não precisa ter rodado
as anteriores nesta sessão.

In [ ]:
RUN_A = r"""COLE_AQUI_O_BLOCO_DA_PRIMEIRA_GPU"""
RUN_B = r"""COLE_AQUI_O_BLOCO_DA_SEGUNDA_GPU"""

# ──────────────────────────────────────────────────────────────────────
import json
A, B = json.loads(RUN_A.strip()), json.loads(RUN_B.strip())

def h(s):
    return float.fromhex(s)

def rel(a, b):
    a, b = h(a), h(b)
    d = abs(a - b)
    return d / max(abs(a), abs(b), 1e-300)

print("=" * 72)
for r in (A, B):
    i = r["info"]
    print("%-26s cap %-5s %2d SMs | torch %s cu%s cudnn %s | tf32=%s"
          % (i["gpu"], i["capability"], i["sms"], i["torch"], i["cuda"],
             i["cudnn"], i["tf32"]))
print("=" * 72)

for r, n in ((A, "A"), (B, "B")):
    if not r["mesma_gpu_identica"]:
        print("AVISO: a run %s nem se reproduziu na propria GPU." % n)

if A["info"]["gpu"] == B["info"]["gpu"]:
    print("AVISO: os dois blocos sao da MESMA GPU. Troque o ambiente e repita.")
print()

# ── onde a divergencia nasce ──────────────────────────────────────────
print("FASE A — forward unico, pesos identicos, sem treino")
print("-" * 72)
print("  %-34s %-12s %s" % ("camada", "erro rel.", "veredito"))
pior = 0.0
for k in A["fase_a"]:
    if k not in B["fase_a"]:
        continue
    d = rel(A["fase_a"][k], B["fase_a"][k])
    pior = max(pior, d)
    print("  %-34s %-12.3e %s" % (k, d, "identico" if d == 0 else "difere"))
print()

if pior == 0:
    print("  VEREDITO: as duas GPUs concordam BIT A BIT no forward.")
    print("  A aritmetica nao e fonte de divergencia nesta carga.")
else:
    primeira = next(k for k in A["fase_a"]
                    if k in B["fase_a"] and rel(A["fase_a"][k], B["fase_a"][k]) > 0)
    print("  VEREDITO: divergem. Nasce em '%s'." % primeira)
    print("  Erro relativo maximo no forward: %.3e" % pior)
    print("  (referencia: eps do fp32 = 5,96e-08; do TF32 = 4,88e-04)")
    if pior > 1e-5:
        print("  ATENCAO: grande demais para ordem de reducao. Confira o TF32.")
print()

# ── como cresce durante o treino ──────────────────────────────────────
print("FASE B — crescimento da diferenca ao longo do treino")
print("-" * 72)
n = min(len(A["passos"]), len(B["passos"]))
prim = None
for i in range(n):
    if A["passos"][i] != B["passos"][i]:
        prim = i + 1
        break
if prim is None:
    print("  Os %d primeiros passos sao bit a bit identicos." % n)
else:
    print("  Primeiro passo divergente: passo %d de %d (erro rel. %.3e)"
          % (prim, n, rel(A["passos"][prim - 1], B["passos"][prim - 1])))

print()
print("  %-4s %-13s %-13s %-11s %s" % ("ep", "perda A", "perda B", "erro rel.", "acc A / acc B"))
for ea, eb in zip(A["epocas"], B["epocas"]):
    print("  %-4d %-13.8f %-13.8f %-11.2e %.2f%% / %.2f%%"
          % (ea["ep"], h(ea["perda"]), h(eb["perda"]),
             rel(ea["perda"], eb["perda"]), ea["acc"], eb["acc"]))

da = abs(A["epocas"][-1]["acc"] - B["epocas"][-1]["acc"])
print()
print("  Diferenca de acuracia no fim: %.2f p.p." % da)
print("  (a dispersao entre folds da campanha e de 0,16 a 0,41 p.p.)")
print()

# ── custo de pinar ────────────────────────────────────────────────────
print("FASE C — custo de pinar tudo numa GPU so")
print("-" * 72)
sa, sb = A["seg_epoca"], B["seg_epoca"]
rapida, lenta = (A, B) if sa < sb else (B, A)
razao = max(sa, sb) / min(sa, sb)
print("  %-26s %.3f s/epoca" % (A["info"]["gpu"], sa))
print("  %-26s %.3f s/epoca" % (B["info"]["gpu"], sb))
print("  %s e %.2fx mais rapida." % (rapida["info"]["gpu"], razao))
print()
print("  Com 2 vagas de cada tipo, em unidades da GPU rapida:")
print("    2 + 2 (misto)  = %.2f  vagas equivalentes" % (2 + 2 / razao))
print("    2   (pinado)   = 2.00  vagas equivalentes")
perda_pct = (1 - 2 / (2 + 2 / razao)) * 100
print("  Pinar tudo na %s custa %.0f%% de throughput."
      % (rapida["info"]["gpu"], perda_pct))